In [ ]:
# Install dependencies
!pip install transformers datasets sacrebleu sentencepiece kagglehub accelerate --quiet

import os, random, torch, re
import kagglehub
import numpy as np

from datasets import Dataset
from sacrebleu import corpus_bleu, corpus_chrf

from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

import transformers
print("Transformers version:", transformers.__version__)

# ============================================================
# 1. LOAD + CLEAN DATA
# ============================================================

def clean_text(t):
    if not t:
        return ""
    t = t.strip()
    if len(t) == 0:
        return ""
    if re.fullmatch(r"[^\w]+", t):
        return ""
    return t

dataset_path = kagglehub.dataset_download(
    "parvmodi/english-to-malayalam-machine-translation-dataset"
)

with open(os.path.join(dataset_path, "train.en"), encoding="utf-8") as f:
    english_raw = [clean_text(l) for l in f][:2000]

with open(os.path.join(dataset_path, "train.ml"), encoding="utf-8") as f:
    malayalam_raw = [clean_text(l) for l in f][:2000]

pairs = [(e, m) for e, m in zip(english_raw, malayalam_raw) if e and m]
english, malayalam = zip(*pairs)

# Split
random.seed(42)
idx = list(range(len(english)))
random.shuffle(idx)

train_en = [english[i] for i in idx[:600]]
train_ml = [malayalam[i] for i in idx[:600]]

eval_en = [english[i] for i in idx[600:700]]
eval_ml = [malayalam[i] for i in idx[600:700]]

test_en = [english[i] for i in idx[700:800]]
test_ml = [malayalam[i] for i in idx[700:800]]

print("Loaded data: Train=", len(train_en), "Eval=", len(eval_en), "Test=", len(test_en))

# ============================================================
# 2. LOAD MODEL (NLLB)
# ============================================================

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(
    model_name, src_lang="eng_Latn", tgt_lang="mal_Mlym"
)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

forced_bos_id = tokenizer.convert_tokens_to_ids("mal_Mlym")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Model loaded on", device)

# ============================================================
# 3. PREPROCESS
# ============================================================

MAX_LEN = 64  # reduced for memory safety

def preprocess(batch):
    en = [x.strip() for x in batch["en"]]
    ml = [x.strip() for x in batch["ml"]]

    model_inputs = tokenizer(
        en, max_length=MAX_LEN, padding="max_length", truncation=True
    )

    labels = tokenizer(
        text_target=ml,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True
    )

    model_inputs["labels"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
        for seq in labels["input_ids"]
    ]

    return model_inputs

train_ds = Dataset.from_dict({"en": train_en, "ml": train_ml}).map(
    preprocess, batched=True, remove_columns=["en", "ml"]
)

eval_ds = Dataset.from_dict({"en": eval_en, "ml": eval_ml}).map(
    preprocess, batched=True, remove_columns=["en", "ml"]
)

print("Dataset preprocessed!")

# ============================================================
# 4. METRICS
# ============================================================

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu = corpus_bleu(decoded_preds, [decoded_labels])
    return {"bleu": bleu.score}

# Disable wandb
os.environ["WANDB_DISABLED"] = "true"

# ============================================================
# 5. TRAINING SETUP (OOM SAFE)
# ============================================================

print("\nCreating training arguments...")

# >>> Memory-saving settings <<<
model.gradient_checkpointing_enable()
model.config.use_cache = False

args = Seq2SeqTrainingArguments(
    output_dir="nllb_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=1,      # SAFE
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,      # Effective batch = 4
    learning_rate=3e-5,
    predict_with_generate=True,
    logging_steps=50,
    save_strategy="no",
    eval_strategy="no",
    fp16=False,
    bf16=True,                          # MUCH safer on Colab GPUs
    report_to=[],
    generation_max_length=MAX_LEN,
)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Training starting…")
trainer.train()
print("Training finished!")

# ============================================================
# 6. FINAL TEST EVALUATION
# ============================================================

print("\nEvaluating on test set...")
model.eval()
preds = []

for i in range(0, len(test_en), 32):
    batch = tokenizer(
        test_en[i:i+32],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **batch,
            forced_bos_token_id=forced_bos_id,
            num_beams=3,
            max_length=MAX_LEN
        )

    preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

bleu = corpus_bleu(preds, [list(test_ml)])
chrf = corpus_chrf(preds, [list(test_ml)])

print("\n==== FINAL RESULTS ====")
print("BLEU:", bleu.score)
print("chrF:", chrf.score)

# ============================================================
# 7. SAVE RESULTS
# ============================================================

os.makedirs("nllb_results", exist_ok=True)

with open("nllb_results/predictions.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(preds))

with open("nllb_results/references.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(test_ml))

print("\nSaved! Sample:")

for i in range(min(3, len(test_en))):
    print("\nEN:", test_en[i])
    print("PRED:", preds[i])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.8 MB/s eta 0:00:00
Transformers version: 4.57.2
Using Colab cache for faster access to the 'english-to-malayalam-machine-translation-dataset' dataset.
Loaded data: Train= 600 Eval= 100 Test= 100


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset preprocessed!

Creating training arguments...
Training starting…


/tmp/ipython-input-783918511.py:167: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
50,2.227500
100,2.019500
150,2.087500
200,1.707900
250,1.722300
300,1.762900
350,1.593500
400,1.545400
450,1.552600


Training finished!

Evaluating on test set...

==== FINAL RESULTS ====
BLEU: 2.857877418943135
chrF: 35.092665886421194

Saved! Sample:

EN: Energy, banking, metal and auto stock led the losses.
PRED: ഊർജ്ജം, ബാങ്കിംഗ്, മെറ്റൽ, ഓട്ടോ സ്റ്റോക്ക് എന്നിവയാണ് നഷ്ടത്തിന് കാരണമായത്.

EN: Jehovah The Happy God
PRED: സന്തുഷ്ടനായ ദൈവം യഹോവ

EN: Twitter users shared similar experiences while commenting to the video.
PRED: ട്വിറ്റർ ഉപയോക്താക്കളും സമാനമായ അനുഭവങ്ങൾ പങ്കുവെച്ചിരുന്നു.


In [ ]:
# ============================================================
# 8. FULL NLP METRIC EVALUATION (BLEU, chrF, ROUGE, METEOR, BERTScore optional)
# ============================================================

print("\nRunning full metric evaluation...")

# Install missing packages if needed
try:
    from rouge_score import rouge_scorer
    from nltk.translate.meteor_score import meteor_score
    from bert_score import score as bert_score
    import nltk
except:
    !pip install rouge-score nltk bert-score --quiet
    from rouge_score import rouge_scorer
    from nltk.translate.meteor_score import meteor_score
    from bert_score import score as bert_score
    import nltk

# FIX NLTK TOKENIZER ERROR
nltk.download("punkt")
nltk.download("punkt_tab")   # <--- THIS FIXES LookupError
nltk.download("wordnet")


def evaluate_all_metrics(references, hypotheses, use_bertscore=False):

    refs_formatted = [references]

    # BLEU
    bleu = corpus_bleu(hypotheses, refs_formatted).score

    # chrF
    chrf = corpus_chrf(hypotheses, refs_formatted).score

    # ROUGE
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=False
    )

    r1 = r2 = rL = 0
    for ref, hyp in zip(references, hypotheses):
        score = scorer.score(ref, hyp)
        r1 += score["rouge1"].fmeasure
        r2 += score["rouge2"].fmeasure
        rL += score["rougeL"].fmeasure

    n = len(hypotheses)
    r1 /= n
    r2 /= n
    rL /= n

    # METEOR -- tokenize first
    meteor_vals = []
    for ref, hyp in zip(references, hypotheses):
        ref_tok = nltk.word_tokenize(ref)
        hyp_tok = nltk.word_tokenize(hyp)
        meteor_vals.append(meteor_score([ref_tok], hyp_tok))

    meteor_avg = sum(meteor_vals) / n

    # BERTScore
    if use_bertscore:
        print("Running BERTScore... (may take 1–3 minutes)")
        P, R, F1 = bert_score(hypotheses, references, lang="en")
        bert_f1 = float(F1.mean())
    else:
        bert_f1 = None

    return {
        "BLEU": bleu,
        "chrF": chrf,
        "ROUGE-1": r1,
        "ROUGE-2": r2,
        "ROUGE-L": rL,
        "METEOR": meteor_avg,
        "BERTScore_F1": bert_f1
    }


# RUN EVALUATION
all_scores = evaluate_all_metrics(
    references=list(test_ml),
    hypotheses=preds,
    use_bertscore=False
)

print("\n==== FULL METRIC RESULTS ====")
for k, v in all_scores.items():
    print(f"{k}: {v}")

# Save metrics
import json
os.makedirs("nllb_results", exist_ok=True)
with open("nllb_results/metrics.json", "w", encoding="utf-8") as f:
    json.dump(all_scores, f, indent=4, ensure_ascii=False)

print("\nMetrics saved to nllb_results/metrics.json")



Running full metric evaluation...


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



==== FULL METRIC RESULTS ====
BLEU: 2.857877418943135
chrF: 35.092665886421194
ROUGE-1: 0.06333333333333332
ROUGE-2: 0.015
ROUGE-L: 0.06333333333333332
METEOR: 0.17425640617219076
BERTScore_F1: None

Metrics saved to nllb_results/metrics.json


In [ ]:
%reset -f
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()